# Phase 1 Dataset Analysis — Medical Cost / Insurance (`insurance`)

This notebook loads the saved Phase 1 pipeline outputs for the **Medical Cost Personal Dataset (`insurance`)** and documents the analyst's manual insights.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

DATASET = 'insurance'
OUT_DIR = Path('..') / 'outputs' / DATASET
CLEAN_CSV = Path('..') / 'data' / 'cleaned' / f'{DATASET}_cleaned.csv'

if OUT_DIR.exists():
    prof = json.loads((OUT_DIR / 'profile.json').read_text(encoding='utf-8'))
    display(pd.DataFrame(prof['columns'])[['name', 'raw_dtype', 'semantic_type', 'pct_missing', 'n_unique', 'semantic_type_reason']])
    display(Markdown((OUT_DIR / 'cleaning_log.md').read_text(encoding='utf-8')))
    display(Markdown((OUT_DIR / 'scaling_report.md').read_text(encoding='utf-8')))
else:
    print(f'Run `ads run --input data/raw/insurance/insurance.csv --name insurance` to populate {OUT_DIR}.')

## Manual Insights (Written by the Analyst)

<!-- MANUAL:START -->
- **Strongest primary effect (`smoker -> charges`):** Smokers incur dramatically higher medical charges (mean ~$32,050 vs. ~$8,434 for non-smokers), creating a distinct multimodal right tail in `charges`.
- **Non-linear interaction (`smoker x bmi -> charges`):** Among non-smokers, higher `bmi` increases `charges` only modestly; however, among smokers, crossing the obesity threshold (`bmi >= 30`) triggers a steep step-change jump in medical charges (often exceeding $35,000-$60,000).
- **Monotonic baseline drift (`age -> charges`):** Within each smoking/BMI stratum, `charges` increases linearly with `age` along three parallel bands corresponding to baseline plan/comorbidity tiers.
- **Duplicate row caveat (`drop_duplicates: auto`):** The raw dataset contains 1 exact duplicate row (`age=19, sex=male, bmi=30.59, children=0, smoker=no, region=northwest, charges=1639.5631`). Because `insurance.csv` has no unique `customer_id`, our conservative policy flags narrow/ID-less duplicate rows carefully while logging exact duplicates.
- **Distributional & scaling implications:** `charges` is strongly right-skewed (`skew > 1.5`) with positive support (`min >= $1,121.87`), making `log1p` transformation ideal for visualization and linear regression, while Phase 3 should prefer Mann-Whitney U / Kruskal-Wallis and Spearman rank correlation.
- **Reference findings for Phase 5 evaluation to recover:** (1) `smoker <-> charges` (large positive effect), (2) `age <-> charges` (positive monotonic correlation), (3) `bmi <-> charges` (positive correlation, amplified by `smoker`), and (4) `children <-> charges` (small positive effect).
<!-- MANUAL:END -->